# Pandas vs PySpark

Notebook ini menerjemahkan operasi pandas yang sudah kamu kenal ke Spark.
Semua syntax yang dipakai di notebook ETL (`02_etl_superstore.ipynb`) dibahas di sini.

Data: Superstore, berupa 3 file CSV di folder `data/`:

| File | Isi |
| --- | --- |
| `superstore_orders.csv` | satu baris per item order: tanggal, ID customer, ID produk, lokasi, sales |
| `superstore_customers.csv` | ID customer, nama, segment |
| `superstore_products.csv` | ID produk, nama, kategori |

**Sebagian syntax pandas ternyata bisa langsung dipakai di Spark.** Tanda di notebook ini:

| Tanda | Arti |
| --- | --- |
| ✅ **Bisa** | syntax pandas ini jalan juga di Spark |
| ⚠️ **Bisa, hasil beda** | jalan, tapi hasilnya tidak sama dengan pandas |
| ❌ **Tidak bisa** | error di Spark, pakai versi Spark yang ditunjukkan |

### Apa itu Spark?
- pandas bekerja di **1 komputer** dan menyimpan semua data di memori.
- Spark memecah data menjadi **partisi** dan memprosesnya **paralel**, sehingga bisa mengolah data yang terlalu besar untuk pandas.
- Kodenya sangat mirip pandas. Dua perbedaan terbesar:
  1. Spark itu **lazy**: baru menghitung saat kita meminta hasil (`show`, `count`, `write`).
  2. DataFrame Spark **tidak bisa diubah langsung**. Hasilnya selalu disimpan: `df = df.withColumn(...)`.

### SETUP

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

os.environ['PYSPARK_PYTHON'] = sys.executable   # Spark memakai Python dari env conda ini

java_home = os.path.join(sys.prefix, 'Library' if os.name == 'nt' else os.path.join('lib', 'jvm'))
if 'JAVA_HOME' not in os.environ and os.path.isdir(java_home):
    os.environ['JAVA_HOME'] = java_home          # pakai Java dari env conda (penting saat dijalankan di VS Code)

spark = (SparkSession.builder
         .appName("pandas-vs-spark")
         .master("local[2]")                           # jalan di laptop, pakai 2 core
         .config("spark.driver.memory", "1g")          # RAM maksimal 1 GB
         .config("spark.sql.shuffle.partitions", "4")  # saat groupBy/join, data dibagi jadi 4 bagian
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")   # sembunyikan pesan warning Spark

Penjelasan setting di `SparkSession.builder`:

| Setting | Arti |
| --- | --- |
| `master("local[2]")` | Spark jalan di laptop sendiri dengan 2 core. Di cluster, isinya alamat cluster |
| `spark.driver.memory` | RAM maksimal yang boleh dipakai Spark |
| `spark.sql.shuffle.partitions` | Jumlah bagian data saat `groupBy`/`join`. Default 200, terlalu banyak untuk data kecil |
| `PYSPARK_PYTHON` | Memastikan Spark memakai Python dari env conda yang aktif |
| `JAVA_HOME` | Lokasi Java dari env conda. Diisi otomatis kalau kosong, misalnya saat notebook dijalankan dari VS Code |

- Saat pertama dijalankan, muncul beberapa baris `WARNING` ketika Spark start. Itu normal.
- `SparkSession` adalah pintu masuk Spark, seperti `import pandas as pd`.
- `functions as F` berisi fungsi-fungsi kolom Spark (`F.col`, `F.sum`, `F.trim`, ...).
- `Window` dipakai untuk membuat nomor urut (bagian 7).
- Selama session berjalan, Spark UI bisa dibuka di http://localhost:4040.

## 1. Membaca & melihat data

❌ pandas `pd.read_csv('file.csv')` → Spark: `spark.read.csv('file.csv', header=True, inferSchema=True, escape='"')`

- `header=True`: baris pertama adalah nama kolom (di pandas otomatis).
- `inferSchema=True`: Spark menebak tipe data tiap kolom (di pandas otomatis).
- `escape='"'`: beberapa teks berisi koma dan tanda kutip, misalnya `Chair, "Executive Red"`.
  pandas menanganinya otomatis; Spark butuh opsi ini, kalau tidak kolomnya bergeser.
- Path `../data/...` artinya folder `data` yang ada di sebelah folder `notebooks`.

In [ ]:
orders = spark.read.csv('../data/superstore_orders.csv', header=True, inferSchema=True, escape='"')

**Tampilkan beberapa baris**

⚠️ pandas `df.head()` **bisa, tapi hasilnya beda**: bukan tabel, melainkan daftar `Row`.

In [ ]:
orders.head(2)

Untuk melihat data sebagai tabel, pakai `show()`:

In [ ]:
orders.show(5)

### Kenapa harus `show()`?

Di pandas, menulis `df` di akhir cell langsung menampilkan tabel. Di Spark, menulis `orders` saja **tidak menampilkan data**,
hanya nama kolom dan tipenya:

In [ ]:
orders

Ini karena Spark **lazy**: DataFrame hanyalah "rencana", datanya belum dibaca atau dihitung.
Hal yang sama terjadi pada filter tanpa `show()`: hasilnya DataFrame baru, tapi belum ada data yang diproses.

In [ ]:
orders[orders['Sales'] > 3000]

Data baru diproses saat kita memanggil **action**, misalnya:

| Action | Hasil |
| --- | --- |
| `show(n)` | menampilkan n baris sebagai teks |
| `count()` | jumlah baris |
| `toPandas()` | mengubah ke DataFrame pandas (hanya untuk data kecil) |
| `write...` | menyimpan ke file / database |

In [ ]:
orders[orders['Sales'] > 3000][['Order ID', 'Sales']].show()

### Tampil otomatis seperti pandas

Spark bisa diatur supaya DataFrame langsung tampil sebagai tabel di Jupyter tanpa `show()`:

In [ ]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)     # tampil otomatis
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 5)     # maksimal 5 baris

orders

Praktis untuk eksplorasi, tapi perlu diingat:
- Setiap kali DataFrame tampil, Spark tetap menjalankan perhitungan di belakang layar.
- Ini hanya berlaku di Jupyter. Di script `.py` (misalnya yang dijalankan Airflow) tidak ada tampilan otomatis, jadi tetap pakai `show()`.

Di notebook ini kita kembali memakai `show()`, supaya sama dengan cara kerja di script:

In [ ]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", False)

**Jumlah baris dan kolom**

- ❌ pandas `df.shape` dan `len(df)` → Spark: `df.count()` (jumlah baris) + `len(df.columns)`
- ✅ pandas `df.columns` **bisa**

In [ ]:
print(orders.count(), len(orders.columns))
print(orders.columns)

**Tipe data tiap kolom**

- ✅ pandas `df.dtypes` **bisa** (hasilnya daftar `(nama kolom, tipe)`)
- ❌ pandas `df.info()` → Spark: `df.printSchema()`

Selalu cek ini setelah membaca file: kolom angka harus `double`/`int`, bukan `string`.
`Order Date` masih `string`, nanti diubah jadi tanggal (bagian 4).

In [ ]:
orders.dtypes[:4]

In [ ]:
orders.printSchema()

## 2. Mengganti nama semua kolom

❌ pandas `df.columns = [...]` dan `df.rename(columns={...})` → Spark: `df.toDF(*nama_baru)`

Nama kolom dengan spasi (`Order ID`) merepotkan, terutama saat masuk ke database. Kita ubah ke **snake_case** (`order_id`).
Karena dipakai untuk 3 file, kita buat fungsi.

In [ ]:
def to_snake_case(df):
    new_names = [c.lower().replace(' ', '_').replace('-', '_').replace('/', '_') for c in df.columns]
    return df.toDF(*new_names)

orders = to_snake_case(orders)
orders.columns

Mulai sekarang semua kolom memakai nama snake_case.

## 3. Memilih, memfilter, mengurutkan

**Memilih kolom** ✅ pandas `df[['order_id', 'sales']]` **bisa**

Versi Spark: `df.select('order_id', 'sales')`. Hasilnya sama.

In [ ]:
orders[['order_id', 'region', 'sales']].show(3)
orders.select('order_id', 'region', 'sales').show(3)   # versi Spark

**Filter, 1 kondisi** ✅ pandas `df[df['sales'] > 3000]` **bisa**

Versi Spark: `df.filter(F.col('sales') > 3000)`.

In [ ]:
orders[orders['sales'] > 3000][['order_id', 'state', 'sales']].show()

**Filter, beberapa kondisi** ✅ pandas `df[(df['sales'] > 1000) & (df['region'] == 'West')]` **bisa**. Setiap kondisi wajib diberi kurung.

In [ ]:
orders[(orders['sales'] > 1000) & (orders['region'] == 'West')][['order_id', 'region', 'sales']].show(5)

**Filter dengan teks** ❌ pandas `df.query('sales > 3000')` → Spark: `df.filter('sales > 3000')`

In [ ]:
orders.filter('sales > 3000').count()

**Mengurutkan** ❌ pandas `df.sort_values('sales', ascending=False)` → Spark: `df.sort('sales', ascending=False)`

Mirip, hanya namanya `sort`. Versi Spark lain: `df.orderBy(F.desc('sales'))`.

In [ ]:
orders.sort('sales', ascending=False)[['order_id', 'product_id', 'sales']].show(5)

**Frekuensi tiap nilai** ❌ pandas `df['region'].value_counts()` → Spark: `df.groupby('region').count()`

`groupby` (huruf kecil, seperti pandas) dan `groupBy` sama saja di Spark.

In [ ]:
orders.groupby('region').count().sort('count', ascending=False).show()

**Nilai unik** ❌ pandas `df['region'].unique()` → Spark: `df[['region']].distinct()`

In [ ]:
orders[['region']].distinct().show()

`WEST` dan `West` adalah region yang sama tapi ditulis berbeda. Ini data kotor yang dibersihkan di bagian 4.

## 4. Membersihkan data

**Menambah / mengubah kolom**

- ❌ pandas `df['unit_price'] = df['sales'] / df['quantity']` → Spark: `df = df.withColumn('unit_price', ...)`
- ✅ operasi antar kolom `df['sales'] / df['quantity']` **bisa**
- ❌ pandas `.round(2)` → Spark: `F.round(kolom, 2)`

`withColumn(nama, nilai)` menambah kolom baru, atau mengganti kolom yang namanya sudah ada.
Hasilnya **harus disimpan** ke variabel, karena DataFrame Spark tidak bisa diubah langsung.

In [ ]:
with_price = orders.withColumn('unit_price', F.round(orders['sales'] / orders['quantity'], 2))
with_price[['sales', 'quantity', 'unit_price']].show(3)

Di tengah rangkaian perintah (chaining), DataFrame-nya belum punya nama variabel, jadi kolom ditulis dengan `F.col('nama')`.
Contoh yang sama dengan `F.col`:

In [ ]:
orders.withColumn('unit_price', F.round(F.col('sales') / F.col('quantity'), 2))[['unit_price']].show(3)

### Duplikat

- ❌ pandas `df.duplicated().sum()` → Spark: `df.count() - df.drop_duplicates().count()`
- ✅ pandas `df.drop_duplicates()` **bisa** (versi Spark: `dropDuplicates()`)

In [ ]:
print('baris duplikat:', orders.count() - orders.drop_duplicates().count())

orders_unique = orders.drop_duplicates()
print(orders.count(), '->', orders_unique.count())

✅ pandas `df.drop_duplicates(subset=['kolom'])` **bisa**: menyisakan **1 baris per nilai kolom itu**.
Contoh: di file products, beberapa `product_id` punya 2 nama produk.

In [ ]:
products = to_snake_case(spark.read.csv('../data/superstore_products.csv', header=True, inferSchema=True, escape='"'))

products.groupby('product_id').agg(F.countDistinct('product_name').alias('names')).filter('names > 1').show(3)
print(products.count(), '->', products.drop_duplicates(subset=['product_id']).count())

### Nilai kosong

- ❌ pandas `df.isna().sum()` dan `df['kolom'].isna()` → Spark: `df['kolom'].isNull()`
- ✅ filter `df[df['kolom'].isNull()]` **bisa**
- ✅ pandas `df.fillna({...})` dan `df.dropna(subset=[...])` **bisa**, syntax sama

In [ ]:
for c in orders.columns:
    n = orders[orders[c].isNull()].count()
    if n > 0:
        print(c, n)

In [ ]:
filled = orders.fillna({'postal_code': 0})     # isi nilai kosong
dropped = orders.dropna(subset=['sales'])      # buang baris yang sales-nya kosong
print(orders.count(), '->', dropped.count())

### Teks

❌ pandas `.str.strip().str.title()` → Spark: `F.initcap(F.trim(kolom))`

**Jebakan:** `df['region'].str.strip()` atau `df['city'].str.contains(...)` **tidak langsung error** saat ditulis,
tapi error saat datanya dihitung (`show`/`count`). Di Spark tidak ada `.str`.

In [ ]:
clean_region = orders.withColumn('region', F.initcap(F.trim('region')))
clean_region[['region']].distinct().show()

`F.trim` membuang spasi di awal/akhir, `F.initcap` membuat huruf pertama tiap kata kapital (`WEST` → `West`).

### Ubah tipe data

- ✅ pandas `.astype('string')` **bisa** (tipe ditulis sebagai teks; `.astype(str)` ❌ tidak bisa). Versi Spark: `.cast('string')`
- ❌ pandas `.str.len()` → Spark: `F.length(kolom)`
- ❌ pandas `.str.zfill(5)` → Spark: `F.lpad(kolom, 5, '0')`

Postal code terbaca sebagai angka, jadi `06824` berubah menjadi `6824` (angka 0 di depan hilang).

In [ ]:
zip_fixed = orders.withColumn('postal_code', F.lpad(orders['postal_code'].astype('string'), 5, '0'))

orders[F.length(orders['postal_code'].astype('string')) < 5][['city', 'postal_code']].show(3)
zip_fixed[zip_fixed['city'] == 'Fairfield'][['city', 'postal_code']].show(3)

### Tanggal

- ❌ pandas `pd.to_datetime(kolom, format='%m/%d/%Y')` → Spark: `F.to_date(kolom, 'M/d/yyyy')`
- ❌ pandas `.dt.year`, `.dt.quarter`, `.dt.month` → Spark: `F.year()`, `F.quarter()`, `F.month()`.
  **Jebakan:** sama seperti `.str`, `.dt.year` tidak langsung error, tapi error saat `show()`.
- ❌ pandas `.dt.strftime(...)` → Spark: `F.date_format(kolom, pola)`

Pola tanggal Spark: `M` = bulan, `d` = hari, `yyyy` = tahun.

In [ ]:
dated = orders.withColumn('order_date', F.to_date('order_date', 'M/d/yyyy'))

(dated.select('order_date',
              F.year('order_date').alias('year'),
              F.quarter('order_date').alias('quarter'),
              F.month('order_date').alias('month'),
              F.date_format('order_date', 'MMMM').alias('month_name'),
              F.date_format('order_date', 'EEEE').alias('day_name'),
              F.date_format('order_date', 'yyyyMMdd').cast('int').alias('date_id'))
      .show(3))

`select(...)` di sini dipakai karena kolom-kolomnya hasil fungsi; `df[[...]]` hanya untuk memilih kolom yang sudah ada. `.alias()` memberi nama kolom hasil.

## 5. Group by & agregasi

- ✅ pandas `df.groupby('region').agg({'sales': 'sum'})` **bisa**, tapi ⚠️ nama kolom hasilnya `sum(sales)`
- ❌ pandas `df.groupby('region')['sales'].sum()` dan `.agg(total=('sales', 'sum'))` → Spark: `.agg(F.sum('sales').alias('total'))`

In [ ]:
orders.groupby('region').agg({'sales': 'sum'}).show()

Versi Spark lebih fleksibel: banyak agregasi sekaligus, nama kolom bebas, dan bisa dibulatkan:

In [ ]:
(clean_region.groupby('region')
    .agg(F.round(F.sum('sales'), 2).alias('total_sales'),
         F.count('order_id').alias('rows'),
         F.countDistinct('customer_id').alias('customers'))
    .sort('total_sales', ascending=False)
    .show())

## 6. Menggabungkan data dari beberapa file (join)

`orders` hanya berisi `product_id` dan `customer_id`. Nama produk, kategori, dan nama customer ada di file lain.

❌ pandas `pd.merge(orders, products, on=...)` / `orders.merge(...)` → Spark: `orders.join(products, on='product_id', how='left')`
(pandas juga punya `df.join`, jadi bentuk ini tetap familiar.)

In [ ]:
customers = to_snake_case(spark.read.csv('../data/superstore_customers.csv', header=True, inferSchema=True, escape='"'))

orders_products = orders.join(products, on='product_id', how='left')
orders_products[['order_id', 'product_id', 'product_name', 'category', 'sales']].show(3, truncate=False)

**Join 3 tabel**: sambungkan `.join()` berurutan, seperti menyambung `pd.merge()`.

Perhatikan jumlah barisnya: hasil join **lebih banyak** dari `orders`. File customers dan products berisi ID duplikat,
dan join mengulang order untuk setiap baris yang cocok. Selalu cek jumlah baris setelah join, dan bersihkan duplikat sebelum join.

In [ ]:
full = (orders
        .join(customers, on='customer_id', how='left')
        .join(products, on='product_id', how='left'))
print('orders:', orders.count(), '| setelah join:', full.count())

full_clean = (orders
              .join(customers.drop_duplicates(subset=['customer_id']), on='customer_id', how='left')
              .join(products.drop_duplicates(subset=['product_id']), on='product_id', how='left'))
print('setelah duplikat dibersihkan:', full_clean.count())

**Jenis join** (sama dengan `how=` di pandas dan JOIN di SQL):

| `how=` | Yang disimpan |
| --- | --- |
| `'inner'` | hanya baris yang cocok di kedua tabel |
| `'left'` | semua baris tabel kiri |
| `'left_anti'` | baris tabel kiri yang **tidak punya** pasangan (untuk mencari data yang hilang; tidak ada di pandas) |

`left_anti`: order yang produknya tidak ada di file products.

In [ ]:
orders.join(products, on='product_id', how='left_anti')[['order_id', 'product_id']].show()

`inner`: hanya order yang produknya ada. Dua order di atas ikut terbuang.

In [ ]:
valid = orders.join(products.drop_duplicates(subset=['product_id'])[['product_id']], on='product_id', how='inner')
print(orders.count(), '->', valid.count())

Join juga bisa memakai **beberapa kolom** sekaligus: `on=['state', 'city']` (sama seperti pandas).
Baris dianggap cocok kalau semua kolomnya sama.

## 7. Menumpuk baris & membuat nomor urut

**Menumpuk 2 DataFrame** ❌ pandas `pd.concat([a, b])` → Spark: `a.union(b)`. Kolomnya harus sama.

In [ ]:
order_dates = dated.select(F.col('order_date').alias('full_date'))
ship_dates = orders.select(F.to_date('ship_date', 'M/d/yyyy').alias('full_date'))

all_dates = order_dates.union(ship_dates).distinct()
print(order_dates.count(), '+', ship_dates.count(), '-> unik:', all_dates.count())

**Nomor urut 1, 2, 3, ...** ❌ pandas `df['id'] = range(1, len(df) + 1)` → Spark: `F.row_number().over(Window.orderBy(...))`

Memberi nomor urut sesuai urutan kolom yang dipilih. Dipakai untuk membuat ID bagi data yang belum punya ID.

In [ ]:
cities = orders[['state', 'city']].drop_duplicates()
cities_with_id = cities.withColumn('city_id', F.row_number().over(Window.orderBy('state', 'city')))
cities_with_id.show(5)

## Ringkasan: syntax pandas di Spark

| pandas | Di Spark | Versi Spark |
| --- | --- | --- |
| `pd.read_csv(path)` | ❌ | `spark.read.csv(path, header=True, inferSchema=True, escape='"')` |
| `df.head()` | ⚠️ hasilnya list `Row` | `df.show(5)` |
| `df.shape`, `len(df)` | ❌ | `(df.count(), len(df.columns))` |
| `df.columns`, `df.dtypes` | ✅ | sama |
| `df.info()` | ❌ | `df.printSchema()` |
| `df.columns = [...]`, `df.rename(...)` | ❌ | `df.toDF(*nama_baru)` |
| `df[['a', 'b']]` | ✅ | `df.select('a', 'b')` |
| `df[df['a'] > 1]`, kondisi `&` / `\|` | ✅ | `df.filter(F.col('a') > 1)` |
| `df.query('a > 1')` | ❌ | `df.filter('a > 1')` |
| `df['a'].isin([...])` | ✅ | sama |
| `df.sort_values('a', ascending=False)` | ❌ | `df.sort('a', ascending=False)` |
| `df['a'].value_counts()` | ❌ | `df.groupby('a').count()` |
| `df['a'].unique()` | ❌ | `df[['a']].distinct()` |
| `df['new'] = ...` | ❌ | `df = df.withColumn('new', ...)` |
| `df['a'] / df['b']` | ✅ | sama |
| `.round(2)` | ❌ | `F.round(kolom, 2)` |
| `df.drop_duplicates()`, `(subset=[...])` | ✅ | `df.dropDuplicates([...])` |
| `df.duplicated()` | ❌ | `df.count() - df.drop_duplicates().count()` |
| `df.isna()`, `df['a'].isna()` | ❌ | `df['a'].isNull()` |
| `df.fillna({...})`, `df.dropna(subset=[...])` | ✅ | sama |
| `.str.strip()`, `.str.title()`, `.str.contains()` | ❌ (error saat `show`) | `F.trim`, `F.initcap`, `F.col('a').contains(...)` |
| `.astype('string')` | ✅ (`.astype(str)` ❌) | `.cast('string')` |
| `.str.len()`, `.str.zfill(5)` | ❌ | `F.length()`, `F.lpad(..., 5, '0')` |
| `pd.to_datetime(...)` | ❌ | `F.to_date('a', 'M/d/yyyy')` |
| `.dt.year`, `.dt.strftime(...)` | ❌ (error saat `show`) | `F.year()`, `F.date_format()` |
| `df.groupby('a').agg({'b': 'sum'})` | ✅ ⚠️ kolom jadi `sum(b)` | `.agg(F.sum('b').alias('total'))` |
| `df.groupby('a')['b'].sum()` | ❌ | `df.groupby('a').agg(F.sum('b'))` |
| `pd.merge(a, b, on=, how=)` | ❌ | `a.join(b, on=, how=)` |
| `pd.concat([a, b])` | ❌ | `a.union(b)` |
| `df['id'] = range(...)` | ❌ | `F.row_number().over(Window.orderBy(...))` |
| `df.to_sql(...)`, `pd.read_sql(...)` | ❌ | `df.write.jdbc(...)`, `spark.read.jdbc(...)` (di notebook ETL) |

Selesai. Hentikan Spark supaya memori laptop dilepas.

In [ ]:
spark.stop()